# 02 — Feature Engineering

Run all 8 dimension extractors on iHUT data and inspect feature outputs.
Calibrate score normalizer reference corpus.

In [ ]:
import sys
sys.path.insert(0, '../')

import pandas as pd
from pathlib import Path
from pipeline.ingestion.ihut_extractor import extract_multiple_ihut_pdfs
from pipeline.features.base_extractor import Review
from pipeline.features.feature_registry import DIMENSION_EXTRACTORS, DIMENSION_DISPLAY_NAMES
from pipeline.scoring.score_aggregator import compute_vibe_score

# Load iHUT PDFs
pdf_files = list(Path('../../').glob('*.pdf'))
all_docs = extract_multiple_ihut_pdfs(pdf_files)

# Convert verbatim chunks to Review objects
reviews = []
for doc in all_docs:
    for chunk in doc.verbatim_chunks:
        reviews.append(Review(text=chunk.text, source='ihut', market=chunk.market))

all_chunks = [chunk for doc in all_docs for chunk in doc.chunks]
print(f'Total reviews (iHUT verbatims): {len(reviews)}')

In [ ]:
# Run each extractor and inspect features
for extractor in DIMENSION_EXTRACTORS:
    result = extractor.extract(reviews, all_chunks)
    print(f'\n=== {DIMENSION_DISPLAY_NAMES[result.dimension]} (weight: {result.weight*100:.0f}%) ===')
    print(f'  Raw score:      {result.raw_score:.4f}')
    print(f'  Weighted raw:   {result.weighted_raw:.4f}')
    print(f'  Signal count:   {result.feature_count}')
    print(f'  Feature vector: {result.feature_vector}')
    if result.top_signals:
        print(f'  Top signal: {result.top_signals[0].signal_type} = "{result.top_signals[0].signal_value}"')

In [ ]:
# Compute full VIBE score
vibe = compute_vibe_score(reviews, all_chunks)
print(f'\nVIBE Score: {vibe.vibe_score} / 100')
print(f'Band: {vibe.score_band}')
print(f'Confidence: {vibe.confidence}')
print(f'\nSHAP breakdown:')
print(f'  Baseline: {vibe.baseline_score}')
for dim, shap_val in sorted(vibe.shap_values.items(), key=lambda x: -abs(x[1])):
    sign = '+' if shap_val >= 0 else ''
    print(f'  {DIMENSION_DISPLAY_NAMES[dim]:25s}: {sign}{shap_val:.2f} pts')
print(f'  {"Final Score":25s}: {vibe.vibe_score}')